# Session 18 — Train-Test Split, Dataset Preparation, ML Pipeline Overview
### Module 5: Supervised Machine Learning

This notebook covers:
- Why we split data into training and testing sets
- Using `train_test_split()` in practice
- Stratified splitting for classification problems
- Preparing a dataset for modeling: separating features/target, encoding, scaling (overview)
- Building a basic end-to-end pipeline with `sklearn.pipeline.Pipeline`
- Hands-on practice exercises

We continue using the **California Housing** (regression) and **Breast Cancer Wisconsin** (classification) datasets from the previous session, so we can build directly on what we already explored.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split


## 1. Quick Recap

- **Regression** predicts a continuous number (e.g. house value)
- **Classification** predicts a fixed category (e.g. malignant/benign)
- Both are *supervised* — the model learns from labeled examples (X, y)

Today's question: **once we have labeled data, how do we actually use it to build and fairly evaluate a model?**


## 2. Why Do We Need a Train-Test Split?

If we train a model on *all* our data and then test it on the *same* data, we're not really testing anything — the model has already seen the answers. This can make a model look far more accurate than it actually is on new, unseen data.

**The solution:** split the data into two parts before training:
- **Training set** — used to teach the model (fit it to the data)
- **Testing set** — held back, used only to check how well the model performs on data it has never seen

This is the single most important habit in supervised machine learning, and skipping it is one of the most common beginner mistakes.

> **Analogy:** think of the training set as practice questions with an answer key, and the test set as the actual exam. If you "practice" using the exam questions themselves, your score won't tell you how well you actually learned the material.


## 3. Loading and Preparing the Datasets


In [2]:
housing = fetch_california_housing()
df_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
df_housing['MedHouseValue'] = housing.target
df_housing.head()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseValue
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
cancer = load_breast_cancer()
df_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df_cancer['target'] = cancer.target   # 0 = malignant, 1 = benign
df_cancer.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## 4. Separating Features (X) and Target (y)

Before splitting, we separate the **input features (X)** from the **target/label (y)** we want to predict.


In [4]:
# Regression dataset
X_reg = df_housing.drop(columns=['MedHouseValue'])   # all columns except the target
y_reg = df_housing['MedHouseValue']                   # target only

print("X_reg shape:", X_reg.shape)
print("y_reg shape:", y_reg.shape)


X_reg shape: (20640, 8)
y_reg shape: (20640,)


In [5]:
# Classification dataset
X_clf = df_cancer.drop(columns=['target'])
y_clf = df_cancer['target']

print("X_clf shape:", X_clf.shape)
print("y_clf shape:", y_clf.shape)


X_clf shape: (569, 30)
y_clf shape: (569,)


## 5. Using `train_test_split()`

`train_test_split()` from `sklearn.model_selection` randomly divides X and y into training and testing portions, keeping the corresponding rows of X and y aligned.


In [6]:
from sklearn.model_selection import train_test_split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg,
    test_size=0.2,      # 20% of the data reserved for testing
    random_state=42      # ensures the same split every time we run this cell
)

print("Training set size:", X_train_reg.shape)
print("Testing set size:", X_test_reg.shape)


Training set size: (16512, 8)
Testing set size: (4128, 8)


### Key parameters of `train_test_split()`

| Parameter | Purpose |
|---|---|
| `test_size` | Proportion (or count) of data reserved for testing, e.g. `0.2` = 20% |
| `train_size` | Alternative way to specify the training proportion |
| `random_state` | A fixed seed so the split is reproducible every time the code runs |
| `shuffle` | Whether to shuffle data before splitting (default `True`) — important since datasets are often ordered |
| `stratify` | Ensures class proportions are preserved in both sets — critical for classification |


## 6. Stratified Splitting for Classification

For classification problems, a plain random split can sometimes produce a training or testing set with a very different class balance than the original data — especially with smaller or imbalanced datasets. `stratify` fixes this by preserving the class proportions in both splits.


In [7]:
print("Original class balance:")
print(y_clf.value_counts(normalize=True))


Original class balance:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64


In [8]:
# WITHOUT stratify
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)
print("Test set class balance (no stratify):")
print(y_test_a.value_counts(normalize=True))


Test set class balance (no stratify):
target
1    0.622807
0    0.377193
Name: proportion, dtype: float64


In [9]:
# WITH stratify
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print("Test set class balance (with stratify):")
print(y_test_clf.value_counts(normalize=True))


Test set class balance (with stratify):
target
1    0.631579
0    0.368421
Name: proportion, dtype: float64


**Notice:** with `stratify=y_clf`, the proportion of each class in the test set closely matches the original dataset. This is considered best practice for classification problems, especially when classes are imbalanced.


## 7. How Much Data Should Go to Testing?

There's no single "correct" split, but common choices are:

| Split | When to use |
|---|---|
| 80% train / 20% test | Most common default, works well for medium-to-large datasets |
| 70% train / 30% test | When you want a larger, more reliable test set |
| 90% train / 10% test | When the dataset is very large and 10% is still plenty of test data |

**General rule:** more training data usually helps the model learn better, but the test set must still be large enough to give a reliable, representative performance estimate.


## 8. Dataset Preparation — Beyond Just Splitting

Before (or as part of) building a model, real datasets typically need further preparation. We introduce these here at a conceptual level — feature scaling and categorical encoding will be covered in full detail in a later session (Session 25).


### 8.1 Handling Categorical Columns (Preview)

Most ML algorithms require numeric input. Categorical (text-based) columns must be converted to numbers before modeling — for example using one-hot encoding.


In [10]:
# Example using a small sample categorical column (not part of our main datasets)
sample = pd.DataFrame({'city': ['Chennai', 'Delhi', 'Mumbai', 'Chennai']})
pd.get_dummies(sample, columns=['city'])


,city_Chennai,city_Delhi,city_Mumbai
0,True,False,False
1,False,True,False
2,False,False,True
3,True,False,False


In [19]:
'''| Method                | What it does                                                                                    | Numerical Example                                                                                                         |
| --------------------- | ----------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------- |
| **`fit()`**           | Learns the mean and standard deviation from the training data. It does **not** change the data. | Training data: **[10, 20, 30]** → Mean = **20**, Std ≈ **8.16**                                                           |
| **`transform()`**     | Uses the learned mean and standard deviation to scale the data.                                 | Test data: **[15, 25]** → Uses Mean = **20** and Std = **8.16** → Scaled values ≈ **[-0.61, 0.61]**                       |
| **`fit_transform()`** | Performs both `fit()` and `transform()` in one step.                                          | Training data: **[10, 20, 30]** → Learns Mean = **20**, Std = **8.16**, then scales to approximately **[-1.22, 0, 1.22]** |'''

'| Method                | What it does                                                                                    | Numerical Example                                                                                                         |\n| --------------------- | ----------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------- |\n| **`fit()`**           | Learns the mean and standard deviation from the training data. It does **not** change the data. | Training data: **[10, 20, 30]** → Mean = **20**, Std ≈ **8.16**                                                           |\n| **`transform()`**     | Uses the learned mean and standard deviation to scale the data.                                 | Test data: **[15, 25]** → Uses Mean = **20** and Std = **8.16** → Scaled values ≈ **[-0.61, 0.61]**                       |\n| *

### 8.2 Feature Scaling (Preview)

Some algorithms (like KNN, covered in Session 21) are sensitive to the *scale* of features — a column ranging from 0-1 and another ranging from 0-100,000 can unfairly dominate distance-based calculations. Scaling brings features to a comparable range.


In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# IMPORTANT: fit the scaler on the TRAINING data only, then apply it to both sets
X_train_scaled = scaler.fit_transform(X_train_reg)
X_test_scaled = scaler.transform(X_test_reg)

print("Before scaling -- mean of first feature:", X_train_reg.iloc[:, 0].mean().round(2))
print("After scaling -- mean of first feature:", X_train_scaled[:, 0].mean().round(2))


Before scaling -- mean of first feature: 3.88
After scaling -- mean of first feature: -0.0


**Critical rule:** always `fit` preprocessing steps (like scalers or encoders) on the **training data only**, then use `transform` (not `fit_transform`) on the test data. Fitting on the full dataset before splitting leaks information from the test set into training — this is called **data leakage** and produces misleadingly optimistic results.


## 9. The Machine Learning Pipeline — Overview

A typical supervised learning pipeline follows this sequence:

1. **Collect data** — gather the raw dataset
2. **Explore & clean** — the EDA and cleaning steps from Module 4
3. **Separate features and target** — X and y
4. **Split into train and test sets** — using `train_test_split()`
5. **Preprocess** — scale numeric features, encode categorical features (fit on train, apply to both)
6. **Train the model** — fit the chosen algorithm on the training data
7. **Evaluate the model** — check performance on the test data (Module 6)
8. **Predict on new data** — use the trained model in the real world

Steps 5 and 6 are often combined using an `sklearn` **Pipeline**, so preprocessing and modeling happen together, consistently, every time.


## 10. Building a Basic Pipeline

`sklearn.pipeline.Pipeline` chains preprocessing steps and a model together into a single object. This keeps preprocessing consistent between training and prediction, and helps prevent data leakage.


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# A pipeline with two steps: scaling, then a regression model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# Fitting the pipeline automatically fits the scaler AND the model on the training data
pipeline.fit(X_train_reg, y_train_reg)

# Predicting automatically applies the SAME fitted scaler before predicting
predictions = pipeline.predict(X_test_reg)
print("First 5 predictions:", predictions[:5])
print("First 5 actual values:", y_test_reg.values[:5])


First 5 predictions: [0.71912284 1.76401657 2.70965883 2.83892593 2.60465725]
First 5 actual values: [0.477   0.458   5.00001 2.186   2.78   ]


**Why use a pipeline instead of doing these steps manually?**
- Less code duplication between training and prediction
- Impossible to accidentally forget to transform new data the same way
- Cleaner, more professional, more reproducible code — especially valuable in the capstone project later in this course


## 11. Putting It All Together — A Mini End-to-End Example


In [13]:
from sklearn.linear_model import LogisticRegression

# Step 1-2: data already loaded and explored (df_cancer)
# Step 3: features/target already separated (X_clf, y_clf)

# Step 4: train-test split (stratified, since this is classification)
X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Step 5-6: preprocessing + model combined in a pipeline
clf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000))
])
clf_pipeline.fit(X_train, y_train)

# Step 7: a quick peek at performance (full evaluation techniques come in Module 6)
accuracy = clf_pipeline.score(X_test, y_test)
print(f"Test accuracy: {accuracy:.3f}")


Test accuracy: 0.982


> Full evaluation metrics (accuracy, precision, recall, F1-score, confusion matrix) are covered in depth in **Module 6**. For now, `.score()` gives us a quick, single-number sanity check.


## 12. Hands-on Practice Exercise 1 — Basic Splitting

Using `df_housing`:

1. Separate features (X) and target (y), where the target is `MedHouseValue`
2. Split the data into 75% training and 25% testing using `train_test_split()`
3. Print the shape of all four resulting objects
4. Change `random_state` to a different number and re-run — confirm the split changes


In [14]:
# Write your solution here



## 13. Hands-on Practice Exercise 2 — Stratified Splitting

Using `df_cancer`:

1. Split the data into 80% training and 20% testing WITHOUT `stratify`
2. Split the data again WITH `stratify=y`
3. Compare the class balance of `y_test` in both cases using `.value_counts(normalize=True)`
4. Write 1-2 sentences explaining why stratification matters more on imbalanced datasets


In [15]:
# Write your solution here



## 14. Hands-on Practice Exercise 3 — Build a Pipeline

Using `df_housing`:

1. Separate features and target, then split into train/test sets (80/20)
2. Build a `Pipeline` with a `StandardScaler` step and a `LinearRegression` step
3. Fit the pipeline on the training data
4. Use the pipeline to predict on the test data and print the first 5 predictions vs actual values
5. In a markdown cell, explain in your own words why we fit the scaler only on training data


In [16]:
# Write your solution here



## Key Terms Glossary

| Term | Meaning |
|---|---|
| Training set | Data used to fit/teach the model |
| Testing set | Held-out data used to evaluate the model on unseen examples |
| `train_test_split()` | sklearn function that splits X and y into train/test sets |
| `test_size` | Proportion of data reserved for testing |
| `random_state` | Seed value for reproducible splits |
| `stratify` | Preserves class proportions across train/test splits |
| Data leakage | When information from the test set improperly influences training |
| `fit_transform()` vs `transform()` | Fit learns parameters from data; transform applies already-learned parameters |
| `Pipeline` | Chains preprocessing and modeling steps into a single reusable object |
| One-hot encoding | Converting categorical columns into numeric 0/1 columns |
| Feature scaling | Bringing numeric features to a comparable range |


## Common Mistakes to Avoid

- Evaluating a model on the same data it was trained on, producing an overly optimistic (and misleading) performance score
- Forgetting `stratify=y` on an imbalanced classification dataset, leading to a test set that doesn't represent the true class balance
- Fitting a scaler or encoder on the *entire* dataset before splitting — this leaks test set information into training
- Using `fit_transform()` on the test set instead of `transform()` (this refits the scaler on test data, which is incorrect)
- Not setting `random_state`, making results impossible to reproduce or fairly compare across experiments
- Assuming a bigger test set is always better — too large a test set leaves too little data for the model to learn from


## Homework / Practice Assignment

1. Using either `df_housing` or `df_cancer`, perform a full data preparation workflow: separate X/y, split into train/test (stratified if classification), and build a `Pipeline` with at least one preprocessing step and one model
2. Try 3 different `test_size` values (e.g. 0.1, 0.2, 0.3) and observe how the resulting train/test shapes change
3. In writing, explain what data leakage is and describe one concrete way it could accidentally happen in a real project
4. Come prepared with 2 questions/doubts for the next session on Linear Regression: concept, implementation, and prediction
